# El Niño 2026

Under normal conditions, Trade Winds blow from east to west across the Equatorial Pacific. 

During an El Niño event, the atmospheric pressure gradient weakens, causing these trade winds to slow down or even reverse into westerlies. Without the wind stress holding it back, the warm water from the western Pacific propagates eastward toward South America. 

In this notebook, we will analyze the zonal (east-west) wind component in the central Pacific to directly observe the breakdown of the trade winds during the ongoing 2026 El Niño event. To detect the El Niño signal, we must isolate the zonal wind component (`eastward_wind`) rather than looking at total wind speed. Normally, the trade winds blow strongly from east to west, which registers as large negative values in our dataset. During a strong El Niño event, this atmospheric circulation collapses. We expect to see these negative zonal wind speeds approach zero, or even turn positive (blowing west to east) in the central Pacific, indicating a complete reversal of the typical flow.

In [ ]:
# Import libraries
import copernicusmarine
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.dates as mdates
from pathlib import Path

In [ ]:
#Here you need to put your credentials once you register
copernicusmarine.login(
    username="",
    password=""
)

In [ ]:
# Download the data directly from https://marine.copernicus.eu/register-copernicus-marine-service (you need an account). 
# Bounding box covers the central Equatorial Pacific for the past 2 years.

copernicusmarine.subset(
    dataset_id="cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H",
    variables=["eastward_wind"],
    minimum_longitude=-180, 
    maximum_longitude=-80,
    minimum_latitude=-5,
    maximum_latitude=5,
    start_datetime="2025-01-01T00:00:00",
    end_datetime="2026-10-01T00:00:00",
    output_directory="data"
)

In [ ]:
# To visualize this collapse of trade winds, we will process our downloaded data into two distinct plots. First, we calculate the monthly average of the zonal
# wind within the equatorial range (5°S to 5°N).

# Time Series: We plot the spatial mean across the entire central Pacific bounding box. We are looking for the overall zonal wind speed to rise 
# from deep negative values (strong westward trade winds) toward zero or above.
# Hovmöller Diagram: By averaging the data only across latitudes, we can plot longitude on the X-axis and time on the Y-axis. This
# allows us to watch exactly where and when the trade wind collapse (warm colors) propagates across the Pacific.

data_time = xr.open_mfdataset('data/*.nc')

# Monthly Average 
u_wind_monthly = data_time['eastward_wind'].resample(time='1MS').mean().squeeze()

# Time Series: Spatial mean across the entire domain (longitude and latitude)
ts_u_wind = u_wind_monthly.mean(dim=['longitude', 'latitude'])

# Hovmöller Diagram: Meridional average (mean along latitude)
hovmoller_data = u_wind_monthly.mean(dim='latitude')

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [1, 2]})
fig.subplots_adjust(hspace=0.3)

# --- Subplot 1: Time Series ---
ax1.plot(ts_u_wind.time, ts_u_wind.values, marker='o', color='b', linewidth=2)
ax1.axhline(0, color='black', linestyle='--', alpha=0.5)
ax1.set_title('Equatorial Zonal Wind Time Series (5°S - 5°N Mean)', fontsize=14)
ax1.set_ylabel('Zonal Wind Speed [m/s]')
ax1.grid(True, alpha=0.3)

# --- Subplot 2: Hovmöller Diagram ---
hov = hovmoller_data.plot.pcolormesh(
    ax=ax2,
    x='longitude', 
    y='time',
    cmap='RdBu_r', 
    vmin=-6, vmax=6,
    cbar_kwargs={'label': 'Zonal Wind [m/s]\n(Blue = Trade Winds, Red = Westerlies)'}
)

ax2.set_title('Hovmöller Diagram: Propagation of Trade Wind', fontsize=14)
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Time')

plt.show()

In [ ]:
# As discussed in class, the effect of El Niño is visible as an interannual variation in the Global Mean Sea Level (GMSL).
# Copernicus Marine Service provides a pre-computed climate indicator for this exact purpose: "Global Ocean Mean Sea Level time series and trend 
# from Observations Reprocessing".
# Let's download and plot this dataset:

# Download
copernicusmarine.subset(
    dataset_id="omi_climate_sl_global_area_averaged_anomalies",
    start_datetime="1999-02-20",
    end_datetime="2026-10-01",
    output_directory="data_gmsl"
)

In [ ]:
# Load the downloaded NetCDF file
nc_files = list(Path('data_gmsl').rglob('*.nc'))
ds_gmsl = xr.open_mfdataset(??)

# Plot the curve
fig, ax = plt.subplots(figsize=(14, 5))

var_name = list(ds_gmsl.data_vars)[0]
ds_gmsl[??].plot(ax=ax, color='darkblue', linewidth=2)

ax.set_title('Global Mean Sea Level Anomaly (1993 - Early 2026)', fontsize=14)
ax.set_ylabel('GMSL Anomaly [m]')
ax.set_xlabel('Year')

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right')

ax.grid(True, linestyle='--', alpha=0.6)

plt.show()

In this pre-computed reprocessed product, the data stops in early 2026, even though today is October 2026. To extend the time coverage and capture the ongoing El Niño event, we must download the gridded Sea Level Anomaly (SLA) data ourselves and compute the global average manually.

We will use Level-4 (L4) globally gridded SLA data. The Copernicus Marine Service provides two complementary products that we can combine:
* Multi-Year (MY) Reprocessed: Offers calibrated monthly mean data from January 1, 1993, to May 15, 2026.
* Near Real-Time (NRT): Covers the remaining gap from mid-May 2026 to the present day (but is provided at a daily temporal resolution).

In the next steps, we will download both datasets. To safely combine them, we will compute the global average for each dataset, align the temporal resolution by resampling the daily NRT data into monthly averages, and finally merge them into a continuous 33-year time series.

NOTE: The SLA spatial means must be weighted by the cosine of the latitude because the physical area of the grid cells decreases as you move from the equator toward the poles.

In [ ]:
# Data download
 
copernicusmarine.subset(
    dataset_id="cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1M-m",
    variables=["sla"],
    start_datetime="1993-01-01T00:00:00",
    end_datetime="2026-05-14T23:59:59",
    output_directory="data_sla_monthly/my"
)

copernicusmarine.subset(
    dataset_id="cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D",
    variables=["sla"],
    start_datetime="2026-05-15T00:00:00",
    end_datetime="2026-10-01T00:00:00",
    output_directory="data_sla_monthly/nrt"
)

In [ ]:
# Load the datasets
ds_my = xr.open_mfdataset('data_sla_monthly/my/*.nc')
ds_nrt = xr.open_mfdataset('data_sla_monthly/nrt/*.nc')

# Calculate spatial means

weights_my = ??
weights_my.name = "weights"
gmsl_my = ds_my['sla'].weighted(weights_my).mean(dim=['longitude', 'latitude']).squeeze()

weights_nrt = ??
weights_nrt.name = "weights"
gmsl_nrt = ds_nrt['sla'].weighted(weights_nrt).mean(dim=['longitude', 'latitude']).squeeze()

# Resample the daily NRT time series to monthly to match the MY data
gmsl_nrt_monthly = gmsl_nrt.resample(time='1MS').mean()

# Concatenate the resulting 1D time series
gmsl_raw = xr.concat([??, ??], dim='time')
gmsl_raw = gmsl_raw.sortby('time').drop_duplicates(dim='time')


In [ ]:
# Monthly Mean and Quadratic Trend 

# From the scientific literature we know that the Mean Sea Level rise is accelerating (see lecture slides), therefore we fit a quadratic polynomial

# Calculate quadratic trend
quadratic_fit = gmsl_raw.polyfit(dim='time', deg=2)
trend_curve = xr.polyval(gmsl_raw.time, quadratic_fit.polyfit_coefficients)

# Extract acceleration and rate
decimal_years = gmsl_raw.time.dt.year + gmsl_raw.time.dt.dayofyear / 365.25
centered_years = decimal_years - decimal_years.mean()

# Convert SLA from meters to millimeters
sla_mm = gmsl_raw.values * 1000

# Fit a 2nd degree polynomial: y = ax^2 + bx + c
poly_mm = np.polyfit(centered_years, sla_mm, 2)

# Acceleration is 2 * a (since y = 1/2 * accel * t^2)
acceleration = 2 * poly_mm[0]
# Mean Rate is b
mean_rate = poly_mm[1]

annot_text = f"Mean Rate: {mean_rate:.2f} mm/yr\nAcceleration: {acceleration:.3f} mm/yr²"

# Plot the data
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(??.time, ??.values, color='lightsteelblue', alpha=0.8, 
        label='Monthly SLA')
ax.plot(??.time, ??.values, color='red', linewidth=2, 
        label='Quadratic Trend')

ax.text(0.02, 0.85, annot_text, transform=ax.transAxes, fontsize=12,
        bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray', boxstyle='round,pad=0.5'))

ax.set_title('Global Mean Sea Level Anomaly & Quadratic Trend (1993 - Oct 2026)', fontsize=14)
ax.set_ylabel('GMSL Anomaly [m]')
ax.set_xlabel('Year')

ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right')

ax.legend(loc='lower right')
ax.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [ ]:
# --- Isolating the Interannual Signal ---

# Subtract the long-term quadratic trend
gmsl_detrended = ??

# Apply the 12-month rolling mean to remove the seasonal cycle
gmsl_enso_signal = gmsl_detrended.rolling(time=??, center=True).mean()

# Plot the interannual anomalies
fig, ax = plt.subplots(figsize=(14, 6))

# Fill positive anomalies in red, negative in blue
ax.fill_between(gmsl_enso_signal.time.values, 0, gmsl_enso_signal.values, 
                where=(gmsl_enso_signal.values > 0), color='crimson', alpha=0.6, 
                label='Positive Anomaly')
ax.fill_between(gmsl_enso_signal.time.values, 0, gmsl_enso_signal.values, 
                where=(gmsl_enso_signal.values < 0), color='royalblue', alpha=0.6, 
                label='Negative Anomaly')

ax.plot(??.time, ??.values, color='black', linewidth=1.5)
ax.axhline(0, color='black', linewidth=1)

ax.set_title('Interannual GMSL Anomalies (Quadratic Trend & Seasonality Removed)', fontsize=14)
ax.set_ylabel('Detrended SLA [m]')
ax.set_xlabel('Year')

ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right')

ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.show()

From the previous plot, we can identify the highest positive and negative peaks corresponding to major interannual events:
* Positive Anomalies (El Niño): 1998, 2016, and 2024
* Negative Anomalies (La Niña): 2011 

As of October 2026, the detrended GMSL anomaly is rising again. Current seasonal outlooks forecast a transition into a strong El Niño phase centered between November 2026 and January 2027. *(See the [WMO El Niño/La Niña Update](https://wmo.int/resources/publication-series/el-ninola-nina-updates/august-2026) for the latest updates).*